# Top2Vec: Topic Term Drift — 5-Year Sliding Windows

**Approach:**
1. Aggregate top-words per topic per **5-year window** (ranked by word frequency across years in window)
2. Compute **RBO drift** between consecutive windows as sliding-window drift

**Windows:** `2000–2005 → 2005–2010 → 2010–2015 → 2015–2020 → 2020–2025`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
TEMPORAL_DIR = Path("../../../../results/top2vec/temporal")
RESULT_DIR   = Path("../../../../results/top2vec/consistency")

# 5-year sliding windows (inclusive on both ends)
WINDOWS = [
    (2000, 2005),
    (2005, 2010),
    (2010, 2015),
    (2015, 2020),
    (2020, 2025),
]
TOP_N = 60  # words per window topic
RBO_P = 0.9

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Reading from : {TEMPORAL_DIR}")
print(f"Saving to    : {RESULT_DIR}")
print(f"Windows      : {[f'{s}–{e}' for s,e in WINDOWS]}")

Reading from : ../../../../results/top2vec/temporal
Saving to    : ../../../../results/top2vec/consistency
Windows      : ['2000–2005', '2005–2010', '2010–2015', '2015–2020', '2020–2025']


In [3]:
def parse_words(words_str):
    return [w.strip() for w in str(words_str).split(",") if w.strip()]


def rbo(list1, list2, p=0.9):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    if len(list1) <= len(list2):
        S, L = list1, list2
    else:
        S, L = list2, list1

    s, l = len(S), len(L)
    S_seen, L_seen = set(), set()
    X = 0
    rbo_val = 0.0
    disjoint = 0.0
    ext_term = 0.0

    for d in range(l):
        if d < s:
            s_item = S[d]
            S_seen.add(s_item)
        else:
            s_item = None

        l_item = L[d]
        L_seen.add(l_item)

        overlap_incr = 0
        if d < s:
            if s_item == l_item:
                overlap_incr = 1
            else:
                if s_item in L_seen:
                    overlap_incr += 1
                if l_item in S_seen:
                    overlap_incr += 1
        else:
            if l_item in S_seen:
                overlap_incr = 1

        X += overlap_incr

        if d < s:
            A_d = 2.0 * X / (len(S_seen) + len(L_seen))
        else:
            A_d = X / (d + 1)

        rbo_val += (1 - p) * (p ** d) * A_d

        if d < s:
            ext_term = A_d * (p ** (d + 1))
        else:
            X_s = X - overlap_incr if d == s else X_s
            disjoint += (1 - p) * (p ** d) * (
                X_s * (d + 1 - s) / ((d + 1) * s)
            )
            ext_term = (
                ((X - X_s) / (d + 1) + X_s / s)
                * (p ** (d + 1))
            )

        if p ** d < 1e-12:
            break

    return min(max(rbo_val + disjoint + ext_term, 0.0), 1.0)


def aggregate_window_words(topic_word_map, topic_id, years_in_window, top_n=10):
    """
    Merge top-words for `topic_id` across all `years_in_window`.
    Words ranked by total frequency (how many years they appear in top_words).
    Returns a list of top_n words ordered by frequency desc.
    """
    word_counts = defaultdict(int)
    for y in years_in_window:
        words = topic_word_map.get((y, topic_id), [])
        for w in words:
            word_counts[w] += 1
    if not word_counts:
        return []
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    return [w for w, _ in sorted_words[:top_n]]

## 5-Year Sliding Window Drift

For each topic:
1. Aggregate top-words per 5-year window (by word frequency within that window)
2. Compute RBO drift between consecutive windows

Outputs:
- `window_topwords.csv` — per-topic top-words for each 5-year window
- `window_drift.csv` — per-topic RBO similarity & drift between consecutive windows
- `window_drift_avg.csv` — average drift per window-transition across all topics
- `window_drift_summary.csv` — overall summary per subject

In [4]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"5-Year Sliding Window Drift: {subject.upper()} (Top2Vec)")
    print(f"{'='*70}")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    # Build (year, topic_id) → [words] map
    topic_word_map = {}
    for _, row in evo_df.iterrows():
        key = (int(row["year"]), int(row["topic_id"]))
        topic_word_map[key] = parse_words(row["top_words"])

    all_topics = sorted(evo_df["topic_id"].unique())

    # ── Step 1: aggregate top-words per topic per window ────────────────
    window_topwords_rows = []
    # topic_id → window_label → [words]
    topic_window_words = {}  # key: (topic_id, win_label)

    for tid in all_topics:
        for (win_start, win_end) in WINDOWS:
            years_in_window = list(range(win_start, win_end + 1))
            win_label = f"{win_start}–{win_end}"
            words = aggregate_window_words(topic_word_map, tid, years_in_window, top_n=TOP_N)
            topic_window_words[(tid, win_label)] = words
            window_topwords_rows.append({
                "subject":    subject,
                "topic_id":   tid,
                "window":     win_label,
                "win_start":  win_start,
                "win_end":    win_end,
                "top_words":  ", ".join(words),
                "n_words":    len(words),
            })

    wt_df = pd.DataFrame(window_topwords_rows)
    wt_df.to_csv(RESULT_DIR / subject / "window_topwords.csv", index=False)

    # ── Step 2: compute RBO drift between consecutive windows ────────────
    drift_rows = []
    for tid in all_topics:
        for i in range(len(WINDOWS) - 1):
            win_a_label = f"{WINDOWS[i][0]}–{WINDOWS[i][1]}"
            win_b_label = f"{WINDOWS[i+1][0]}–{WINDOWS[i+1][1]}"
            words_a = topic_window_words.get((tid, win_a_label), [])
            words_b = topic_window_words.get((tid, win_b_label), [])

            if not words_a or not words_b:
                continue  # topic not active in one of the windows

            sim   = rbo(words_a, words_b, p=RBO_P)
            drift = 1.0 - sim

            drift_rows.append({
                "subject":      subject,
                "topic_id":     tid,
                "window_from":  win_a_label,
                "window_to":    win_b_label,
                "transition":   f"{win_a_label} → {win_b_label}",
                "rbo_sim":      round(sim,   6),
                "drift":        round(drift, 6),
                "words_from":   ", ".join(words_a),
                "words_to":     ", ".join(words_b),
            })

    drift_df = pd.DataFrame(drift_rows)
    drift_df.to_csv(RESULT_DIR / subject / "window_drift.csv", index=False)

    # ── Step 3: average drift per window-transition ──────────────────────
    avg_drift = drift_df.groupby(["window_from", "window_to", "transition"]).agg(
        avg_sim   = ("rbo_sim", "mean"),
        avg_drift = ("drift",   "mean"),
        std_drift = ("drift",   "std"),
        n_topics  = ("topic_id", "count"),
    ).reset_index()
    avg_drift.insert(0, "subject", subject)
    avg_drift.to_csv(RESULT_DIR / subject / "window_drift_avg.csv", index=False)

    # ── Step 4: overall summary ──────────────────────────────────────────
    summary = {
        "subject":        subject,
        "n_topics":       len(all_topics),
        "n_windows":      len(WINDOWS),
        "n_transitions":  len(WINDOWS) - 1,
        "avg_drift":      round(drift_df["drift"].mean(), 6),
        "std_drift":      round(drift_df["drift"].std(),  6),
        "median_drift":   round(drift_df["drift"].median(), 6),
        "pct_stable":     round((drift_df["drift"] < 0.4).mean() * 100, 2),
        "pct_moderate":   round(((drift_df["drift"] >= 0.4) & (drift_df["drift"] < 0.7)).mean() * 100, 2),
        "pct_high_drift": round((drift_df["drift"] >= 0.7).mean() * 100, 2),
    }
    pd.DataFrame([summary]).to_csv(RESULT_DIR / subject / "window_drift_summary.csv", index=False)

    # ── Print results ────────────────────────────────────────────────────
    print(f"  Topics: {len(all_topics)} | Windows: {len(WINDOWS)} | Transitions: {len(WINDOWS)-1}")
    print(f"  Overall drift: mean={summary['avg_drift']:.4f} ± {summary['std_drift']:.4f}, "
          f"median={summary['median_drift']:.4f}")
    print(f"  Stable (<0.4): {summary['pct_stable']:.1f}%  "
          f"Moderate (0.4–0.7): {summary['pct_moderate']:.1f}%  "
          f"High (≥0.7): {summary['pct_high_drift']:.1f}%")

    print(f"\n  Average drift per window transition:")
    for _, r in avg_drift.iterrows():
        bar = "█" * int(r["avg_drift"] * 40)
        print(f"    {r['transition']:28s}: sim={r['avg_sim']:.4f}  "
              f"drift={r['avg_drift']:.4f}±{r['std_drift']:.4f}  "
              f"({int(r['n_topics'])} topics)  {bar}")

    print(f"\n  Top 5 most DRIFTED topics (overall avg):")
    topic_avg = drift_df.groupby("topic_id")["drift"].mean().sort_values(ascending=False)
    for tid, d in topic_avg.head(5).items():
        print(f"    T{int(tid):>3} | avg_drift={d:.4f}")

    print(f"\n  Top 5 most STABLE topics (overall avg):")
    for tid, d in topic_avg.tail(5).sort_values().items():
        print(f"    T{int(tid):>3} | avg_drift={d:.4f}")

    print(f"\n  Saved: window_topwords.csv, window_drift.csv, "
          f"window_drift_avg.csv, window_drift_summary.csv")


5-Year Sliding Window Drift: CS (Top2Vec)
  Topics: 309 | Windows: 5 | Transitions: 4
  Overall drift: mean=0.5806 ± 0.2293, median=0.5668
  Stable (<0.4): 24.1%  Moderate (0.4–0.7): 44.5%  High (≥0.7): 31.4%

  Average drift per window transition:
    2000–2005 → 2005–2010       : sim=0.2742  drift=0.7258±0.2020  (214 topics)  █████████████████████████████
    2005–2010 → 2010–2015       : sim=0.3209  drift=0.6791±0.2238  (251 topics)  ███████████████████████████
    2010–2015 → 2015–2020       : sim=0.4344  drift=0.5656±0.1963  (299 topics)  ██████████████████████
    2015–2020 → 2020–2025       : sim=0.5859  drift=0.4141±0.1622  (308 topics)  ████████████████

  Top 5 most DRIFTED topics (overall avg):
    T168 | avg_drift=0.9201
    T291 | avg_drift=0.9122
    T196 | avg_drift=0.8795
    T 65 | avg_drift=0.8626
    T228 | avg_drift=0.8489

  Top 5 most STABLE topics (overall avg):
    T  0 | avg_drift=0.2030
    T  1 | avg_drift=0.2584
    T 46 | avg_drift=0.2716
    T114 | avg_dr

## Endpoint Drift (2000–2005 → 2020–2025)

For each topic: aggregate **all words** across the first window (2000–2005) and last window (2020–2025) using `top_n=60`, then compute RBO drift between them.
This captures **total long-term vocabulary change** from the earliest to the latest 5-year period.

In [5]:
ENDPOINT_TOP_N  = 60   # include all words for a comprehensive comparison
WIN_FIRST_LABEL = "2000–2005"
WIN_LAST_LABEL  = "2020–2025"
WIN_FIRST       = (2000, 2005)
WIN_LAST        = (2020, 2025)

for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Endpoint Drift (2000–2005 → 2020–2025): {subject.upper()} (Top2Vec)")
    print(f"{'='*70}")

    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    topic_word_map = {}
    for _, row in evo_df.iterrows():
        key = (int(row["year"]), int(row["topic_id"]))
        topic_word_map[key] = parse_words(row["top_words"])

    all_topics = sorted(evo_df["topic_id"].unique())
    years_first = list(range(WIN_FIRST[0], WIN_FIRST[1] + 1))
    years_last  = list(range(WIN_LAST[0],  WIN_LAST[1]  + 1))

    ep_rows = []
    for tid in all_topics:
        words_first = aggregate_window_words(
            topic_word_map, tid, years_first, top_n=ENDPOINT_TOP_N
        )
        words_last = aggregate_window_words(
            topic_word_map, tid, years_last, top_n=ENDPOINT_TOP_N
        )

        if not words_first or not words_last:
            continue  # topic not active in one of the endpoint windows

        sim   = rbo(words_first, words_last, p=RBO_P)
        drift = 1.0 - sim

        # Vocabulary overlap analysis
        set_first = set(words_first)
        set_last  = set(words_last)
        kept      = sorted(set_first & set_last)
        removed   = sorted(set_first - set_last)
        added     = sorted(set_last  - set_first)

        ep_rows.append({
            "subject":      subject,
            "topic_id":     tid,
            "window_from":  WIN_FIRST_LABEL,
            "window_to":    WIN_LAST_LABEL,
            "rbo_sim":      round(sim,   6),
            "endpoint_drift": round(drift, 6),
            "n_words_first": len(words_first),
            "n_words_last":  len(words_last),
            "n_kept":        len(kept),
            "n_removed":     len(removed),
            "n_added":       len(added),
            "pct_kept":      round(len(kept) / len(set_first) * 100, 2) if set_first else 0.0,
            "words_first":   ", ".join(words_first),
            "words_last":    ", ".join(words_last),
            "words_kept":    ", ".join(kept),
            "words_added":   ", ".join(added[:20]),   # cap display at 20
            "words_removed": ", ".join(removed[:20]), # cap display at 20
        })

    ep_df = pd.DataFrame(ep_rows)
    ep_df.to_csv(RESULT_DIR / subject / "endpoint_drift.csv", index=False)

    # ── Summary ──────────────────────────────────────────────────────────
    ep_summary = {
        "subject":            subject,
        "n_topics_compared":  len(ep_df),
        "avg_drift":          round(ep_df["endpoint_drift"].mean(), 6),
        "std_drift":          round(ep_df["endpoint_drift"].std(),  6),
        "median_drift":       round(ep_df["endpoint_drift"].median(), 6),
        "avg_pct_kept":       round(ep_df["pct_kept"].mean(), 2),
        "pct_stable":         round((ep_df["endpoint_drift"] < 0.4).mean() * 100, 2),
        "pct_moderate":       round(((ep_df["endpoint_drift"] >= 0.4) & (ep_df["endpoint_drift"] < 0.7)).mean() * 100, 2),
        "pct_high_drift":     round((ep_df["endpoint_drift"] >= 0.7).mean() * 100, 2),
    }
    pd.DataFrame([ep_summary]).to_csv(
        RESULT_DIR / subject / "endpoint_drift_summary.csv", index=False
    )

    # ── Print ─────────────────────────────────────────────────────────────
    print(f"  Topics compared: {ep_summary['n_topics_compared']} "
          f"(topics active in BOTH endpoint windows)")
    print(f"  Endpoint drift : mean={ep_summary['avg_drift']:.4f} ± {ep_summary['std_drift']:.4f}, "
          f"median={ep_summary['median_drift']:.4f}")
    print(f"  Avg vocab kept : {ep_summary['avg_pct_kept']:.1f}%")
    print(f"  Stable  (<0.4) : {ep_summary['pct_stable']:.1f}%  "
          f"Moderate (0.4–0.7): {ep_summary['pct_moderate']:.1f}%  "
          f"High (≥0.7): {ep_summary['pct_high_drift']:.1f}%")

    print(f"\n  Top 5 most DRIFTED (endpoint):")
    for _, r in ep_df.nlargest(5, "endpoint_drift").iterrows():
        print(f"    T{int(r['topic_id']):>3} | drift={r['endpoint_drift']:.4f} | "
              f"kept={r['pct_kept']:.0f}%  "
              f"{r['words_first'][:35]}... → {r['words_last'][:35]}...")

    print(f"\n  Top 5 most STABLE (endpoint):")
    for _, r in ep_df.nsmallest(5, "endpoint_drift").iterrows():
        print(f"    T{int(r['topic_id']):>3} | drift={r['endpoint_drift']:.4f} | "
              f"kept={r['pct_kept']:.0f}%  "
              f"kept words: {r['words_kept'][:60]}")

    print(f"\n  Saved: endpoint_drift.csv, endpoint_drift_summary.csv")



Endpoint Drift (2000–2005 → 2020–2025): CS (Top2Vec)
  Topics compared: 226 (topics active in BOTH endpoint windows)
  Endpoint drift : mean=0.8071 ± 0.1499, median=0.8450
  Avg vocab kept : 15.6%
  Stable  (<0.4) : 1.3%  Moderate (0.4–0.7): 23.0%  High (≥0.7): 75.7%

  Top 5 most DRIFTED (endpoint):
    T 50 | drift=1.0000 | kept=0%  convolution, convolver, directive, ... → accelerator, bit, dnn, hardware, in...
    T107 | drift=1.0000 | kept=0%  controller, evolution, evolutionary... → agent, demonstration, imitation, le...
    T110 | drift=1.0000 | kept=0%  ability, annotated, corpora, great_... → bias, gender, gender_bias, language...
    T196 | drift=1.0000 | kept=0%  camcorder, computer_vision, cyborg_... → endoscopic, robotic, surgeon, surge...
    T219 | drift=1.0000 | kept=0%  adaptation, brill95, cfrule, channe... → learning, mtl, multi, transfer, dat...

  Top 5 most STABLE (endpoint):
    T263 | drift=0.3352 | kept=30%  kept words: algorithm, association, datum, frequent, 

## Inspect Individual Topic

Shows the 5-year window words and drift for a specific topic.

In [6]:
def show_topic_window_drift(topic_id, subject, top_n=10):
    """
    Show 5-year window words and sliding-window drift for a specific topic.
    """
    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")
    topic_evo = evo_df[evo_df["topic_id"] == topic_id]

    if len(topic_evo) == 0:
        print(f"Topic {topic_id} not found in {subject}")
        return

    topic_word_map = {}
    for _, row in topic_evo.iterrows():
        topic_word_map[(int(row["year"]), int(row["topic_id"]))] = parse_words(row["top_words"])

    print(f"\n{'='*70}")
    print(f"Window Drift: Topic {topic_id} — {subject.upper()} (Top2Vec)")
    print(f"{'='*70}")

    win_words = {}
    for (win_start, win_end) in WINDOWS:
        years_in_window = list(range(win_start, win_end + 1))
        win_label = f"{win_start}–{win_end}"
        words = aggregate_window_words(topic_word_map, topic_id, years_in_window, top_n=top_n)
        win_words[win_label] = words
        status = "✓" if words else "✗ (no data)"
        print(f"  [{win_label}] {status}")
        if words:
            print(f"    → {', '.join(words)}")

    print()
    print("  Sliding-window drift (RBO):")
    drifts = []
    for i in range(len(WINDOWS) - 1):
        win_a = f"{WINDOWS[i][0]}–{WINDOWS[i][1]}"
        win_b = f"{WINDOWS[i+1][0]}–{WINDOWS[i+1][1]}"
        wa, wb = win_words.get(win_a, []), win_words.get(win_b, [])
        if not wa or not wb:
            print(f"    {win_a} → {win_b}: N/A (missing window data)")
            continue
        sim   = rbo(wa, wb, p=RBO_P)
        drift = 1.0 - sim
        drifts.append(drift)

        if drift < 0.4:
            label = "🟢 Stable"
        elif drift < 0.7:
            label = "🟡 Moderate"
        else:
            label = "🔴 High drift"

        bar = "█" * int(drift * 30)
        print(f"    {win_a} → {win_b}: sim={sim:.4f}  drift={drift:.4f}  {bar}  {label}")

        added   = set(wb) - set(wa)
        removed = set(wa) - set(wb)
        kept    = set(wa) & set(wb)
        print(f"      Kept ({len(kept)}): {', '.join(sorted(kept)) if kept else '(none)'}")
        if removed:
            print(f"      − Removed: {', '.join(sorted(removed))}")
        if added:
            print(f"      + Added:   {', '.join(sorted(added))}")

    if drifts:
        avg_d = np.mean(drifts)
        if avg_d < 0.4:
            verdict = "🟢 STABLE — vocabulary consistent across time"
        elif avg_d < 0.7:
            verdict = "🟡 MODERATE — topic evolves gradually"
        else:
            verdict = "🔴 HIGH DRIFT — topic vocabulary significantly reshuffles"
        print(f"\n  Avg window drift: {avg_d:.4f}")
        print(f"  Verdict: {verdict}")

In [7]:
# Example: inspect a specific topic
show_topic_window_drift(topic_id=0, subject="cs")


Window Drift: Topic 0 — CS (Top2Vec)
  [2000–2005] ✓
    → algorithm, graph, vertex, edge, planar, polygon, tree, cycle, bhlevel, flipturn
  [2005–2010] ✓
    → algorithm, digraph, edge, graph, planar, polynomial, tree, vertex, approximation, maximum
  [2010–2015] ✓
    → algorithm, approximation, edge, graph, planar, polynomial, tree, vertex, set, point
  [2015–2020] ✓
    → algorithm, edge, graph, planar, set, vertex, approximation, polynomial, tree, subgraph
  [2020–2025] ✓
    → algorithm, edge, graph, planar, polynomial, set, vertex, subgraph, tree, approximation

  Sliding-window drift (RBO):
    2000–2005 → 2005–2010: sim=0.6535  drift=0.3465  ██████████  🟢 Stable
      Kept (6): algorithm, edge, graph, planar, tree, vertex
      − Removed: bhlevel, cycle, flipturn, polygon
      + Added:   approximation, digraph, maximum, polynomial
    2005–2010 → 2010–2015: sim=0.7910  drift=0.2090  ██████  🟢 Stable
      Kept (8): algorithm, approximation, edge, graph, planar, polynomial, t